In [1]:
from pyspark.sql.functions import current_timestamp, col, trim, upper, lower
from utils.spark_utils import create_spark_session

spark = create_spark_session("Silver_Items")

input_path = "s3a://bronze/olist/items"
output_path = "s3a://silver/olist/items"

print(f"Lendo dados da camada Bronze: {input_path}")

Lendo dados da camada Bronze: s3a://bronze/olist/items


In [2]:
df_bronze = spark.read.parquet(input_path)
df_bronze.show(5, truncate=False)
df_bronze.printSchema()

+--------------------------------+-------------+--------------------------------+--------------------------------+-------------------+------+-------------+--------------------------+
|order_id                        |order_item_id|product_id                      |seller_id                       |shipping_limit_date|price |freight_value|data_processamento_bronze |
+--------------------------------+-------------+--------------------------------+--------------------------------+-------------------+------+-------------+--------------------------+
|00010242fe8c5a6d1ba2dd792cb16214|1            |4244733e06e7ecb4970a6e2683c13e61|48436dade18ac8b2bce089ec2a041202|2017-09-19 09:45:35|58.90 |13.29        |2026-07-05 22:57:44.688386|
|00018f77f2f0320c557190d7a144bdd3|1            |e5f2d52b802189ee658865ca93d83a8f|dd7ddc04e1b6c2c614352b383efe2d36|2017-05-03 11:05:13|239.90|19.93        |2026-07-05 22:57:44.688386|
|000229ec398224ef6ca0657da4fc703e|1            |c777355d18b72b67abbeef9df44fd0fd|5b51

In [ ]:
from pyspark.sql.functions import to_timestamp

# Tratamento: trim nos ids, casts numericos (order_item_id -> int, price/freight -> double)
# e conversao da data de limite de envio para timestamp
df_silver = df_bronze \
    .withColumn("order_id", trim(col("order_id"))) \
    .withColumn("order_item_id", col("order_item_id").cast("int")) \
    .withColumn("product_id", trim(col("product_id"))) \
    .withColumn("seller_id", trim(col("seller_id"))) \
    .withColumn("shipping_limit_date", to_timestamp(col("shipping_limit_date"))) \
    .withColumn("price", col("price").cast("double")) \
    .withColumn("freight_value", col("freight_value").cast("double")) \
    .withColumn("dt_criacao_silver", current_timestamp())

df_silver.printSchema()
df_silver.show(5, truncate=False)

In [ ]:
# Conferencia de qualidade: nenhum price deveria virar nulo apos o cast.
# Se aparecer > 0, o formato numerico do CSV precisaria de tratamento extra.
print("price nulos apos cast:", df_silver.filter(col("price").isNull()).count())

# Logica validada aqui foi promovida para silver_rules.tratar_items